# Sub query
<img src="sub_query.png">

## Prepare the data

我们使用 Langchain WebBaseLoader 从博客源加载文档，并通过 RecursiveCharacterTextSplitter 将其拆分为多个片段。

In [1]:
import os

from RAG.Advanced_RAG.rag_utils import sub_query

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a WebBaseLoader instance to load documents from web sources
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        ),# 只解析 HTML 中符合特定条件的部分
    )
)

# Load documents from web sources using the loader
documents=loader.load()

# Initialize a RecursiveCharacterTextSplitter for splitting text into chunks
text_spliiter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

# Split the documents into chunks using the text_splitter
docs=text_spliiter.split_documents(documents)

# Inspect
docs[1]

USER_AGENT environment variable not set, consider setting it to identify your requests.


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool use\n\nThe agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.\n\n\n\n\n\nOverview of a LLM-powered autonomous agent system.')

## Build the chain

We load the docs into milvus vectorstore, and build a milvus retriever.

In [3]:
from rag_utils.vanilla import vectorstore,format_docs,rag_prompt,llm

vectorstore.add_documents(docs)
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

vanilla_rag_chain=(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

Define the sub query chain.

In [5]:
from rag_utils.sub_query import SubQueryRetriever

sub_query_retriever=SubQueryRetriever.from_vectorstore(vectorstore)

sub_query_chain=(
    {"context": sub_query_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

## Test the chain

In [6]:
# MRKL 和 HuggingGPT 有什么区别？
query="What is the difference between MRKL and HuggingGPT?"
vanilla_result=vanilla_rag_chain.invoke(query)
sub_query_result=sub_query_chain.invoke(query)

print(f"\n[vanilla_result]:\n{vanilla_result}\n\n[sub_query_result]:\n{sub_query_result}")

sub_queries: ['Sub-questions:', 'What is MRKL?', 'What is HuggingGPT?']

[vanilla_result]:
Based on the provided context, I can’t give a specific difference because MRKL is not defined or directly compared with HuggingGPT. The context only describes HuggingGPT as using ChatGPT as a task planner to select HuggingFace models and summarize results, while MRKL is mentioned as an earlier workflow reflected in ChemCrow, combining reasoning with tools.

[sub_query_result]:
MRKL is a general neuro-symbolic architecture where an LLM acts as a **router**, sending a request to the most suitable expert module—which may be neural or symbolic, such as a calculator or weather API.

HuggingGPT is a more specific framework where ChatGPT acts as a **task planner**: it parses a user request into multiple tasks, selects appropriate Hugging Face models based on their descriptions, executes the tasks, and then summarizes the results. It has four stages: task planning, model selection, task execution, and re

## 回答质量对比
|方法	|回答内容	|评价|
|---|---|---|
|Vanilla RAG	|"Based on the provided context, I can't give a specific difference..."	|❌ 因为文档里没有直接比较 MRKL 和 HuggingGPT 的段落，普通检索找不到直接答案，模型只能承认不知道
|Sub Query RAG	|分别定义了 MRKL 和 HuggingGPT，并做了对比	|✅ 通过拆解子问题分别检索，找到了两者的单独描述，然后汇总对比，给出了完整的答案|